# BartTorvik EDA

Exploratory notebook for getting familiar with Bart Torvik / T-Rank data and identifying what can support PortalPoint feature engineering.

**Scope for this branch:** BartTorvik-focused EDA. This is intentionally separate from CBBpy/cbbdata work.

**Current approach:** Prefer BartTorvik's direct bulk `.csv`, `.json`, and `.json.gz` files. Avoid HTML scraping unless a needed table is unavailable through a flat-file endpoint.

**Useful references:**
- Bart's data notes: https://adamcwisports.blogspot.com/p/data.html
- `pybart` reference implementation: https://github.com/avewright/pybart
- BartTorvik site: https://barttorvik.com/

**Local requirements:**
- Python 3.11+
- `pandas`, `numpy`, `requests`, `beautifulsoup4`
- Optional: `lxml` or `html5lib` only if we later use `pandas.read_html`


---
## 0. Setup


In [77]:
import csv
import hashlib
import importlib.util
import json
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print('pandas:', pd.__version__)
print('lxml available:', package_available('lxml'))
print('html5lib available:', package_available('html5lib'))

pandas: 2.2.0
lxml available: True
html5lib available: False


In [78]:
# Season convention: 2025 = 2024-25 season.
SEASON = 2025
SAMPLE_TEAM = 'Duke'

BASE_URL = 'https://barttorvik.com/'
CACHE_DIR = Path('../.torvik_cache') if Path.cwd().name == 'notebooks' else Path('.torvik_cache')
CACHE_ENABLED = True
HEADERS = {
    'User-Agent': 'PortalPoint Capstone EDA (educational use; low-volume requests)'
}

def build_url(path: str) -> str:
    return urljoin(BASE_URL, path.lstrip('/'))

def cache_path_for_url(url: str) -> Path:
    digest = hashlib.sha256(url.encode('utf-8')).hexdigest()
    return CACHE_DIR / f'{digest}.txt'

def fetch_text(path: str, params: dict | None = None, timeout: int = 30) -> str:
    """Fetch a BartTorvik flat-file/dynamic endpoint with a light request wrapper."""
    url = build_url(path)
    prepared = requests.Request('GET', url, params=params, headers=HEADERS).prepare()
    cache_path = cache_path_for_url(prepared.url)
    if CACHE_ENABLED and cache_path.exists():
        print('GET', prepared.url, '(cache hit)')
        return cache_path.read_text()

    response = requests.get(url, params=params, headers=HEADERS, timeout=timeout)
    print('GET', response.url)
    print('status:', response.status_code, '| content-type:', response.headers.get('content-type'))
    response.raise_for_status()
    text = response.text
    lowered = text[:500].lower()
    if 'verifying your browser' in lowered or 'cloudflare' in lowered:
        raise RuntimeError('Response looks like a browser verification page, not data.')
    if CACHE_ENABLED:
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(text)
    return text

def dedupe_columns(columns: list[str]) -> list[str]:
    seen = {}
    clean = []
    for col in columns:
        base = str(col).strip() or 'unnamed'
        seen[base] = seen.get(base, 0) + 1
        clean.append(base if seen[base] == 1 else f'{base}_{seen[base]}')
    return clean

def generated_columns(width: int, prefix: str = 'col') -> list[str]:
    return [f'{prefix}_{i:02d}' for i in range(1, width + 1)]

def apply_position_labels(df: pd.DataFrame, labels: dict[int, str]) -> pd.DataFrame:
    """Rename selected 1-based positions while leaving unknown columns positional."""
    df = df.copy()
    rename_map = {df.columns[pos - 1]: name for pos, name in labels.items() if pos <= len(df.columns)}
    return df.rename(columns=rename_map)

def coerce_numeric_columns(df: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        original_non_null = df[col].notna().sum()
        if original_non_null == 0:
            continue
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notna().sum() / original_non_null >= threshold:
            df[col] = converted
    return df

def read_csv_endpoint(path: str, *, has_header: bool = True, params: dict | None = None, prefix: str = 'col') -> pd.DataFrame:
    text = fetch_text(path, params=params)
    rows = list(csv.reader(StringIO(text)))
    if not rows:
        return pd.DataFrame()
    if has_header:
        header = rows[0]
        data = rows[1:]
        width = max([len(header)] + [len(row) for row in data])
        if width > len(header):
            print(f'{path}: header has {len(header)} columns but data has {width}; adding generated extra labels.')
        columns = dedupe_columns(header + [f'extra_{i:02d}' for i in range(1, width - len(header) + 1)])
    else:
        data = rows
        width = max(len(row) for row in data)
        columns = generated_columns(width, prefix=prefix)
    normalized = [row + [None] * (width - len(row)) for row in data]
    return coerce_numeric_columns(pd.DataFrame(normalized, columns=columns))

def read_json_endpoint(path: str, *, columns: list[str] | None = None, params: dict | None = None) -> pd.DataFrame:
    text = fetch_text(path, params=params)
    raw = json.loads(text)
    if isinstance(raw, dict):
        df = pd.DataFrame.from_dict(raw, orient='index').reset_index().rename(columns={'index': 'team'})
    else:
        df = pd.DataFrame(raw)
    if columns is not None and len(df.columns) == len(columns):
        df.columns = dedupe_columns(columns)
    elif columns is not None:
        print(f'{path}: got {len(df.columns)} columns; expected {len(columns)}. Leaving source/positional labels.')
    return coerce_numeric_columns(df)

def df_overview(df: pd.DataFrame, name: str, n: int = 5) -> None:
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(n))
    display(pd.DataFrame({
        'column': df.columns,
        'dtype': [str(dtype) for dtype in df.dtypes],
        'nulls': [int(df[col].isna().sum()) for col in df.columns],
        'null_pct': [float(df[col].isna().mean()) for col in df.columns],
    }))


---
## 1. Endpoint Inventory

Bart's data page says most data is available in direct bulk files and asks people not to run mass scraping against the interactive site. So the EDA priority is:

1. Direct year files like `{year}_team_results.csv`.
2. Lightweight JSON/CSV endpoints like `getadvstats.php?year={year}&csv=1`.
3. HTML scraping only if we cannot find a direct data file.


In [79]:
endpoint_inventory = pd.DataFrame([
    {'dataset': 'team_ratings', 'grain': 'team-season', 'path': f'{SEASON}_team_results.csv', 'format': 'csv', 'header': True, 'priority': 'core'},
    {'dataset': 'team_ratings_json', 'grain': 'team-season', 'path': f'{SEASON}_team_results.json', 'format': 'json', 'header': False, 'priority': 'reference'},
    {'dataset': 'four_factors', 'grain': 'team-season', 'path': f'{SEASON}_fffinal.csv', 'format': 'csv', 'header': True, 'priority': 'core'},
    {'dataset': 'game_results', 'grain': 'game', 'path': f'{SEASON}_results.csv', 'format': 'csv', 'header': False, 'priority': 'core'},
    {'dataset': 'super_schedule', 'grain': 'game', 'path': f'{SEASON}_super_sked.csv', 'format': 'csv', 'header': False, 'priority': 'core'},
    {'dataset': 'master_schedule', 'grain': 'game', 'path': f'{SEASON}_master_sked.csv', 'format': 'csv', 'header': False, 'priority': 'nice-to-have'},
    {'dataset': 'player_stats', 'grain': 'player-season', 'path': 'getadvstats.php', 'format': 'csv', 'header': False, 'priority': 'core'},
    {'dataset': 'player_game_stats', 'grain': 'player-game', 'path': f'{SEASON}_all_advgames.json.gz', 'format': 'json.gz', 'header': False, 'priority': 'core/heavy'},
    {'dataset': 'team_game_log', 'grain': 'team-game', 'path': 'getgamestats.php', 'format': 'json', 'header': False, 'priority': 'core'},
    {'dataset': 'team_shooting_splits', 'grain': 'team-season', 'path': f'{SEASON}_pbp_teamsstats.json', 'format': 'json', 'header': False, 'priority': 'core'},
    {'dataset': 'regular_season_slice', 'grain': 'team-season split', 'path': 'teamslicejson.php', 'format': 'json', 'header': False, 'priority': 'core'},
])

endpoint_inventory


,dataset,grain,path,format,header,priority
0,team_ratings,team-season,2025_team_results.csv,csv,True,core
1,team_ratings_json,team-season,2025_team_results.json,json,False,reference
2,four_factors,team-season,2025_fffinal.csv,csv,True,core
3,game_results,game,2025_results.csv,csv,False,core
4,super_schedule,game,2025_super_sked.csv,csv,False,core
5,master_schedule,game,2025_master_sked.csv,csv,False,nice-to-have
6,player_stats,player-season,getadvstats.php,csv,False,core
7,player_game_stats,player-game,2025_all_advgames.json.gz,json.gz,False,later/heavy
8,team_game_log,team-game,getgamestats.php,json,False,core
9,team_shooting_splits,team-season,2025_pbp_teamsstats.json,json,False,core


---
## 2. Shared Labels for Headerless Files

Some BartTorvik files are headerless. Prefer source-provided headers when they exist. For headerless files, start with positional `*_01` labels and only rename positions that are documented or verified from examples.


In [80]:
GAME_RESULTS_COLS = [
    'game_id', 'date', 'prediction', 'team1_barthag', 'team2_barthag',
    'team1_score', 'team2_score', 'team1_rank', 'team2_rank', 'game_barthag', 'spread'
]

SUPER_SKED_COLS = [
    'muid', 'date', 'conmatch', 'matchup', 'prediction', 'ttq', 'conf', 'venue',
    'team1', 't1oe', 't1de', 't1py', 't1wp', 't1propt',
    'team2', 't2oe', 't2de', 't2py', 't2wp', 't2propt',
    'tpro', 't1qual', 't2qual', 'gp', 'result', 'tempo', 'possessions',
    't1pts', 't2pts', 'winner', 'loser', 't1adjt', 't2adjt',
    't1adjo', 't1adjd', 't2adjo', 't2adjd', 'gamevalue', 'mismatch', 'blowout',
    't1elite', 't2elite', 'ord_date', 't1ppp', 't2ppp', 'gameppp',
    't1rk', 't2rk', 't1gs', 't2gs', 'gamestats', 'overtimes', 't1fun', 't2fun', 'results'
]

MASTER_SKED_COLS = ['game_id', 'date', 'team1_rank', 'team2_rank', 'team1', 'team2']

TEAM_GAME_LOG_COLS = [
    'date', 'neutral', 'team', 'conf', 'opponent', 'venue', 'result',
    'adj_oe', 'adj_de', 'opp_adj_oe', 'efg_pct', 'to_pct', 'or_pct', 'ft_rate',
    'opp_efg_pct', 'opp_to_pct', 'opp_or_pct', 'opp_ft_rate', 'opp_adj_de',
    'opp_conf', 'game_type', 'year', 'tempo', 'game_id', 'coach', 'opp_coach',
    'team_barthag', 'opp_barthag', 'box_score_json', 'extra'
]

PLAYER_STATS_POSITION_LABELS = {
    1: 'player', 2: 'team', 3: 'conf', 4: 'gp', 5: 'min_per',
    6: 'ortg', 7: 'usage', 8: 'efg', 9: 'ts_pct',
    10: 'or_pct', 11: 'dr_pct', 12: 'ast_pct', 13: 'to_pct',
    14: 'ftm', 15: 'fta', 16: 'ft_pct',
    17: 'fg2m', 18: 'fg2a', 19: 'fg2_pct',
    20: 'fg3m', 21: 'fg3a', 22: 'fg3_pct',
    23: 'blk_pct', 24: 'stl_pct', 25: 'ftr',
    26: 'yr', 27: 'ht', 28: 'rec_rank', 29: 'bpm', 30: 'drtg',
    31: 'ast_to_ratio', 32: 'year', 33: 'player_id', 34: 'hometown',
    35: 'rsci', 36: 'min_per_game',
    37: 'rim_made', 38: 'rim_att', 39: 'mid_made', 40: 'mid_att',
    41: 'rim_pct', 42: 'mid_pct', 43: 'dunk_made', 44: 'dunk_att',
    65: 'role', 66: 'role_metric', 67: 'birth_date',
}

PBP_SECTION_COLS = {
    'offense': ['close_2_att', 'close_2_made', 'long_2_att', 'long_2_made', 'three_att', 'three_made', 'ft_att', 'ft_made', 'possessions'],
    'dunks': ['dunk_att', 'dunk_made', 'dunk_share'],
    'defense': ['opp_close_2_att', 'opp_close_2_made', 'opp_long_2_att', 'opp_long_2_made', 'opp_three_att', 'opp_three_made', 'opp_ft_att', 'opp_ft_made', 'opp_possessions'],
    'ddunks': ['opp_dunk_att', 'opp_dunk_made', 'opp_dunk_share'],
}

print('verified player position labels:', len(PLAYER_STATS_POSITION_LABELS))
print('super schedule columns:', len(SUPER_SKED_COLS))


verified player position labels: 47
super schedule columns: 55


---
## 3. Team Ratings

Core team quality baseline: adjusted offense, adjusted defense, Barthag, WAB, projected record, strength of schedule, and tempo.


In [81]:
team_ratings = read_csv_endpoint(f'{SEASON}_team_results.csv')
df_overview(team_ratings, f'team_ratings_{SEASON}')

GET https://barttorvik.com/2025_team_results.csv (cache hit)
team_ratings_2025: 364 rows x 45 columns


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank_2,proj. W,Proj. L,Pro Con W,Pro Con L,Con Rec.,sos,ncsos,consos,Proj. SOS,Proj. Noncon SOS,Proj. Con SOS,elite SOS,elite noncon SOS,Opp OE,Opp DE,Opp Proj. OE,Opp Proj DE,Con Adj OE,Con Adj DE,Qual O,Qual D,Qual Barthag,Qual Games,FUN,ConPF,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Houston,B12,35-5,124.692,11,87.348,1,0.984,1,35.000,5.000,19.000,1.000,19-1,0.774,0.559,0.852,0.774,0.559,0.852,0.586,0.774,115.116,99.471,115.116,99.471,125.380,88.598,126.780,87.840,0.986,21.000,0.012,1442.000,1194.000,1251.450,1.152,0.954,0.000,0.950,11.537,2,146,61.781
1,2,Duke,ACC,35-4,130.568,1,92.014,4,0.982,2,35.000,4.000,19.000,1.000,19-1,0.703,0.647,0.686,0.703,0.647,0.686,0.669,0.672,113.106,101.842,113.106,101.842,130.388,93.736,133.515,93.006,0.985,16.000,0.005,1688.000,1254.000,1315.725,1.283,0.953,0.000,0.950,9.474,5,166,65.963
2,3,Auburn,SEC,32-6,127.683,4,92.974,9,0.975,3,32.000,6.000,15.000,3.000,15-3,0.803,0.661,0.892,0.803,0.661,0.892,0.538,0.649,116.751,99.010,116.751,99.010,128.431,94.349,128.694,92.393,0.978,27.000,0.057,1492.000,1327.000,1262.350,1.182,1.051,0.000,0.833,12.704,1,48,68.287
3,4,Florida,SEC,36-4,127.786,2,94.180,13,0.971,4,36.000,4.000,14.000,4.000,14-4,0.745,0.463,0.877,0.745,0.463,0.877,0.595,0.865,114.913,100.623,114.913,100.623,126.963,94.258,128.527,92.799,0.977,21.000,0.108,1483.000,1289.000,1253.575,1.183,1.028,0.000,0.778,11.146,3,14,69.904
4,5,Alabama,SEC,28-9,127.738,3,96.623,26,0.961,5,28.000,9.000,13.000,5.000,13-5,0.832,0.730,0.889,0.832,0.730,0.889,0.525,0.648,118.217,99.019,118.217,99.019,130.411,97.550,129.304,96.296,0.967,25.000,0.052,1657.000,1509.000,1384.850,1.197,1.090,0.000,0.722,10.117,4,58,75.064


,column,dtype,nulls,null_pct
0,rank,int64,0,0.000
1,team,object,0,0.000
2,conf,object,0,0.000
3,record,object,0,0.000
4,adjoe,float64,0,0.000
5,oe Rank,int64,0,0.000
6,adjde,float64,0,0.000
7,de Rank,int64,0,0.000
8,barthag,float64,0,0.000
9,rank_2,int64,0,0.000


In [82]:
rating_cols = [col for col in [
    'rank', 'team', 'conf', 'record', 'adjoe', 'oe_rank', 'adjde', 'de_rank',
    'barthag', 'barthag_rank', 'wab', 'wab_rank', 'adjt'
] if col in team_ratings.columns]

display(team_ratings[rating_cols].head(20))
display(team_ratings.loc[team_ratings['team'].str.casefold() == SAMPLE_TEAM.casefold(), rating_cols])

,rank,team,conf,record,adjoe,adjde,barthag,adjt
0,1,Houston,B12,35-5,124.692,87.348,0.984,61.781
1,2,Duke,ACC,35-4,130.568,92.014,0.982,65.963
2,3,Auburn,SEC,32-6,127.683,92.974,0.975,68.287
3,4,Florida,SEC,36-4,127.786,94.180,0.971,69.904
4,5,Alabama,SEC,28-9,127.738,96.623,0.961,75.064
5,6,Tennessee,SEC,30-8,120.158,91.878,0.956,63.921
6,7,Texas Tech,B12,28-9,126.063,97.577,0.950,66.138
7,8,Gonzaga,WCC,26-9,124.801,96.709,0.949,70.507
8,9,Maryland,B10,27-9,118.622,92.092,0.948,69.705
9,10,Arizona,B12,24-13,124.918,97.406,0.946,70.275


,rank,team,conf,record,adjoe,adjde,barthag,adjt
1,2,Duke,ACC,35-4,130.568,92.014,0.982,65.963


---
## 4. Player Season Stats

This is one of the most important datasets for PortalPoint. It should support player archetypes, production baselines, shooting profile, usage, efficiency, role, height/class metadata, and possibly transfer matching through `player_id`.

Endpoint: `getadvstats.php?year={SEASON}&csv=1`


In [83]:
player_stats_raw = read_csv_endpoint(
    'getadvstats.php',
    params={'year': SEASON, 'csv': 1},
    has_header=False,
    prefix='player_stat',
)
player_stats = apply_position_labels(player_stats_raw, PLAYER_STATS_POSITION_LABELS)

df_overview(player_stats, f'player_stats_{SEASON}')


GET https://barttorvik.com/getadvstats.php?year=2025&csv=1 (cache hit)
player_stats_2025: 5,060 rows x 67 columns


,player,team,conf,gp,min_per,ortg,usage,efg,ts_pct,or_pct,dr_pct,ast_pct,to_pct,ftm,fta,ft_pct,fg2m,fg2a,fg2_pct,fg3m,fg3a,fg3_pct,blk_pct,stl_pct,ftr,yr,ht,rec_rank,bpm,drtg,ast_to_ratio,year,player_id,hometown,rsci,min_per_game,rim_made,rim_att,mid_made,mid_att,rim_pct,mid_pct,dunk_made,dunk_att,player_stat_45,player_stat_46,player_stat_47,player_stat_48,player_stat_49,player_stat_50,player_stat_51,player_stat_52,player_stat_53,player_stat_54,player_stat_55,player_stat_56,player_stat_57,player_stat_58,player_stat_59,player_stat_60,player_stat_61,player_stat_62,player_stat_63,player_stat_64,role,role_metric,birth_date
0,Robby Carmody,Le Moyne,NEC,24,50.100,109.900,21.000,52.400,58.370,3.800,7.700,8.900,14.300,73,92,0.793,46,91,0.505,34,94,0.362,1.100,2.100,49.700,Sr,6-4,11,1.601,108.052,4.000,2025,65443,"Mars, PA",83.0,0.844,40.000,73.000,6.000,18.000,0.5479,0.3333,1.000,1.000,1.0000,,117.795,125.494,0.551,92.385,-3.840,0.813,-4.653,-3.331,25.140,0.184,-3.516,0.760,1.600,2.360,1.080,0.880,0.240,11.240,Combo G,8.841,1999-05-08
1,Drue Drinnon,Texas St.,SB,24,43.800,109.000,13.700,54.200,58.260,1.100,5.300,14.300,22.700,30,40,0.750,16,39,0.410,24,57,0.421,0.400,1.000,41.700,Sr,6-0,55,0.731,99.354,4.400,2025,65572,"Smyrna, GA",56.8,1.750,9.000,21.000,7.000,18.000,0.4286,0.3889,0.000,0.000,,,115.766,118.213,1.036,63.495,-3.786,-0.187,-3.599,-4.029,22.608,-1.972,-2.058,0.192,1.231,1.423,2.154,0.423,0.077,6.038,Scoring PG,6.344,1999-12-06
2,K.J. Hymes,Nevada,MWC,31,28.900,112.500,16.100,63.500,59.470,12.600,11.700,4.400,15.900,26,55,0.473,40,63,0.635,0,0,0.000,4.700,1.600,87.300,Sr,6-10,42,0.850,109.631,6.100,2025,65575,"Phoenix, AZ",37.4,0.562,38.000,55.000,2.000,8.000,0.6909,0.2500,18.000,19.000,0.9474,,104.820,102.823,1.235,65.671,1.068,-0.043,1.111,-0.754,12.384,-1.199,0.445,1.226,1.161,2.387,0.290,0.323,0.419,3.419,C,0.000,1999-08-10
3,Jack Clark,VCU,A10,34,72.200,127.800,16.000,58.500,61.350,6.900,19.900,12.800,13.800,53,67,0.791,68,105,0.648,49,137,0.358,1.700,2.200,27.700,Sr,6-10,4,3.271,122.399,3.200,2025,65627,"Cheltenham, PA",,1.842,54.000,77.000,14.000,28.000,0.7013,0.5000,15.000,15.000,1.0000,,93.252,92.599,3.929,196.471,6.722,3.898,2.824,6.236,28.780,3.558,2.678,1.771,5.171,6.943,2.000,1.029,0.429,9.657,Stretch 4,8.385,2000-02-14
4,Robert Braswell,Charlotte,Amer,32,61.900,115.500,20.200,55.500,61.300,4.300,8.800,4.400,12.300,90,108,0.833,66,122,0.541,43,113,0.381,3.200,1.800,46.000,Sr,6-7,20,2.164,114.952,4.300,2025,65717,"Jacksonville, FL",69.8,0.485,64.000,103.000,2.000,19.000,0.6214,0.1053,17.000,21.000,0.8095,,112.912,112.503,1.657,124.441,-0.086,1.062,-1.147,0.764,25.064,1.519,-0.755,0.879,1.697,2.576,0.485,0.788,0.697,10.939,Wing G,8.512,1999-10-15


,column,dtype,nulls,null_pct
0,player,object,0,0.000
1,team,object,0,0.000
2,conf,object,0,0.000
3,gp,int64,0,0.000
4,min_per,float64,0,0.000
...,...,...,...,...
62,player_stat_63,float64,2,0.000
63,player_stat_64,float64,2,0.000
64,role,object,0,0.000
65,role_metric,float64,1,0.000


In [84]:
# Columns most likely to matter for early PortalPoint player feature engineering.
player_core_cols = [col for col in [
    'player', 'player_id', 'team', 'conf', 'yr', 'ht', 'role', 'role_metric', 'gp', 'min_per', 'min_per_game',
    'ortg', 'usage', 'efg', 'ts_pct', 'bpm',
    'fg2_pct', 'fg3_pct', 'fg3a', 'rim_att', 'rim_pct', 'mid_att', 'mid_pct',
    'dunk_att', 'dunk_made', 'ast_pct', 'to_pct', 'blk_pct', 'stl_pct',
    'hometown', 'birth_date'
] if col in player_stats.columns]

player_stats[player_core_cols].head(20)


,player,player_id,team,conf,yr,ht,role,role_metric,gp,min_per,min_per_game,ortg,usage,efg,ts_pct,bpm,fg2_pct,fg3_pct,fg3a,rim_att,rim_pct,mid_att,mid_pct,dunk_att,dunk_made,ast_pct,to_pct,blk_pct,stl_pct,hometown,birth_date
0,Robby Carmody,65443,Le Moyne,NEC,Sr,6-4,Combo G,8.841,24,50.100,0.844,109.900,21.000,52.400,58.370,1.601,0.505,0.362,94,73.000,0.5479,18.000,0.3333,1.000,1.000,8.900,14.300,1.100,2.100,"Mars, PA",1999-05-08
1,Drue Drinnon,65572,Texas St.,SB,Sr,6-0,Scoring PG,6.344,24,43.800,1.750,109.000,13.700,54.200,58.260,0.731,0.410,0.421,57,21.000,0.4286,18.000,0.3889,0.000,0.000,14.300,22.700,0.400,1.000,"Smyrna, GA",1999-12-06
2,K.J. Hymes,65575,Nevada,MWC,Sr,6-10,C,0.000,31,28.900,0.562,112.500,16.100,63.500,59.470,0.850,0.635,0.000,0,55.000,0.6909,8.000,0.2500,19.000,18.000,4.400,15.900,4.700,1.600,"Phoenix, AZ",1999-08-10
3,Jack Clark,65627,VCU,A10,Sr,6-10,Stretch 4,8.385,34,72.200,1.842,127.800,16.000,58.500,61.350,3.271,0.648,0.358,137,77.000,0.7013,28.000,0.5000,15.000,15.000,12.800,13.800,1.700,2.200,"Cheltenham, PA",2000-02-14
4,Robert Braswell,65717,Charlotte,Amer,Sr,6-7,Wing G,8.512,32,61.900,0.485,115.500,20.200,55.500,61.300,2.164,0.541,0.381,113,103.000,0.6214,19.000,0.1053,21.000,17.000,4.400,12.300,3.200,1.800,"Jacksonville, FL",1999-10-15
5,Wynston Tabbs,65852,Morgan St.,MEAC,Sr,6-3,Combo G,8.406,10,22.200,1.200,97.800,27.000,39.000,47.270,1.062,0.493,0.132,38,41.000,0.5610,30.000,0.4000,1.000,1.000,18.600,16.400,1.600,2.400,"Suitland, MD",1999-05-22
6,Tyrel Bladen,65878,Norfolk St.,MEAC,Sr,6-10,C,0.000,29,38.400,0.455,98.000,14.900,55.100,55.290,-0.086,0.551,0.000,0,70.000,0.6000,19.000,0.3684,14.000,11.000,5.000,24.000,2.500,1.000,"Coatesville, PA",2000-01-06
7,Jerome Hunter,65892,Xavier,BE,Sr,6-8,Wing F,2.877,34,46.800,0.700,106.600,16.800,56.200,59.880,1.090,0.491,0.531,32,62.000,0.6290,44.000,0.2955,7.000,7.000,7.900,21.400,2.800,1.600,"Columbus, OH",1999-12-22
8,Antwann Jones,65925,Bethune Cookman,SWAC,Sr,6-6,Combo G,10.839,1,0.400,0.000,250.000,7.200,125.000,125.000,2.642,1.000,1.000,1,0.000,,1.000,1.0000,0.000,0.000,0.000,0.000,0.000,0.000,"Orlando, FL",1999-05-29
9,Keenan Fitzmorris,65938,Northwestern,B10,Sr,7-0,C,0.000,22,14.600,1.200,131.700,11.100,63.300,67.630,1.033,0.633,0.000,0,8.000,0.7500,22.000,0.5909,5.000,5.000,5.600,14.000,9.400,0.600,"Overland Park, KS",1999-10-15


In [85]:
# Data quality checks for player-level modeling.
print('Rows:', len(player_stats))
print('Unique players:', player_stats['player_id'].nunique() if 'player_id' in player_stats else 'missing player_id')
print('Duplicate player_id rows:', player_stats.duplicated(subset=['player_id']).sum() if 'player_id' in player_stats else 'missing player_id')
print('Teams:', player_stats['team'].nunique() if 'team' in player_stats else 'missing team')

dq_cols = [col for col in ['player_id', 'player', 'team', 'conf', 'yr', 'ht', 'role', 'role_metric', 'min_per', 'min_per_game', 'usage', 'ortg', 'bpm', 'fg3_pct', 'rim_att', 'birth_date'] if col in player_stats.columns]
pd.DataFrame({
    'column': dq_cols,
    'missing': [int(player_stats[col].isna().sum()) for col in dq_cols],
    'missing_pct': [float(player_stats[col].isna().mean()) for col in dq_cols],
    'unique_values': [int(player_stats[col].nunique(dropna=True)) for col in dq_cols],
})


Rows: 5060
Unique players: 5060
Duplicate player_id rows: 0
Teams: 364


,column,missing,missing_pct,unique_values
0,player_id,0,0.000,5060
1,player,0,0.000,5033
2,team,0,0.000,364
3,conf,0,0.000,31
4,yr,0,0.000,5
5,ht,0,0.000,23
6,role,0,0.000,9
7,role_metric,1,0.000,4245
8,min_per,0,0.000,903
9,min_per_game,2,0.000,3566


In [86]:
# Basic rotation-player filter to reduce tiny-sample noise during exploration.
rotation_players = player_stats.copy()
for col in ['min_per', 'gp', 'usage', 'ortg', 'bpm']:
    if col in rotation_players.columns:
        rotation_players[col] = pd.to_numeric(rotation_players[col], errors='coerce')

rotation_players = rotation_players.query('min_per >= 10 and gp >= 10') if {'min_per', 'gp'} <= set(rotation_players.columns) else rotation_players
print('Rotation-player rows:', len(rotation_players))

leader_cols = [col for col in ['player', 'team', 'conf', 'yr', 'role', 'min_per', 'min_per_game', 'usage', 'ortg', 'bpm', 'ts_pct', 'fg3_pct'] if col in rotation_players.columns]
rotation_players.sort_values('bpm', ascending=False)[leader_cols].head(25)


Rotation-player rows: 3612


,player,team,conf,yr,role,min_per,min_per_game,usage,ortg,bpm,ts_pct,fg3_pct
4747,Bennett Stirtz,Drake,MVC,Jr,Scoring PG,98.800,2.857,26.100,126.400,6.361,60.430,0.396
2112,Bruce Thornton,Ohio St.,B10,Jr,Scoring PG,88.400,3.149,22.000,130.000,6.321,63.300,0.424
330,Ryan Kalkbrenner,Creighton,BE,Sr,C,83.100,1.059,22.300,129.200,6.096,68.430,0.344
152,Eric Dixon,Villanova,BE,Sr,Wing F,84.300,0.893,32.900,116.700,6.090,58.330,0.407
4713,Cooper Flagg,Duke,ACC,Fr,Stretch 4,72.800,2.000,30.800,123.000,5.995,59.260,0.385
1009,Trey Kaufman-Renn,Purdue,B10,Jr,Wing F,76.900,0.987,31.100,118.100,5.742,61.190,0.429
443,Johni Broome,Auburn,SEC,Sr,PF/C,71.400,1.825,30.600,118.500,5.712,54.700,0.278
1685,Braden Smith,Purdue,B10,Jr,Pure PG,92.600,2.872,26.600,116.100,5.704,54.770,0.381
1048,Kam Jones,Marquette,BE,Sr,Scoring PG,83.900,3.175,29.200,118.100,5.667,55.080,0.311
1169,Tyson Degenhart,Boise St.,MWC,Sr,Wing F,84.500,1.135,23.800,126.800,5.616,63.260,0.349


---
## 5. Player ID Stability Across Seasons

For transfer modeling, we need to know whether `player_id` persists across years and schools. This cell pulls a small multi-year slice from the same player stats endpoint.


In [87]:
YEARS_TO_CHECK = [2023, 2024, 2025]

player_years = []
for year in YEARS_TO_CHECK:
    df_raw = read_csv_endpoint(
        'getadvstats.php',
        params={'year': year, 'csv': 1},
        has_header=False,
        prefix='player_stat',
    )
    df = apply_position_labels(df_raw, PLAYER_STATS_POSITION_LABELS)
    df['source_year'] = year
    player_years.append(df)

player_multi_year = pd.concat(player_years, ignore_index=True)
df_overview(player_multi_year, 'player_stats_multi_year', n=3)


GET https://barttorvik.com/getadvstats.php?year=2023&csv=1 (cache hit)
GET https://barttorvik.com/getadvstats.php?year=2024&csv=1 (cache hit)
GET https://barttorvik.com/getadvstats.php?year=2025&csv=1 (cache hit)
player_stats_multi_year: 15,105 rows x 68 columns


,player,team,conf,gp,min_per,ortg,usage,efg,ts_pct,or_pct,dr_pct,ast_pct,to_pct,ftm,fta,ft_pct,fg2m,fg2a,fg2_pct,fg3m,fg3a,fg3_pct,blk_pct,stl_pct,ftr,yr,ht,rec_rank,bpm,drtg,ast_to_ratio,year,player_id,hometown,rsci,min_per_game,rim_made,rim_att,mid_made,mid_att,rim_pct,mid_pct,dunk_made,dunk_att,player_stat_45,player_stat_46,player_stat_47,player_stat_48,player_stat_49,player_stat_50,player_stat_51,player_stat_52,player_stat_53,player_stat_54,player_stat_55,player_stat_56,player_stat_57,player_stat_58,player_stat_59,player_stat_60,player_stat_61,player_stat_62,player_stat_63,player_stat_64,role,role_metric,birth_date,source_year
0,Jailyn Ingram,Georgia,SEC,24,19.600,91.200,19.500,44.400,45.910,2.300,13.800,12.900,17.900,9,15,0.600,7,26,0.269,19,54,0.352,0.800,1.900,18.800,Sr,6-7,15.000,0.222,93.334,4.000,2023,44344,"Madison, GA",,1.000,2.000,10.000,5.000,16.000,0.2000,0.3125,0.000,0.000,,,104.905,101.640,1.005,44.216,-1.733,-1.289,-0.444,-1.663,10.500,-1.643,-0.020,0.208,1.292,1.500,0.625,0.333,0.083,3.333,Wing G,12.548,1997-10-15,2023
1,Deandre Dishman,Middle Tennessee,CUSA,31,68.600,102.800,24.400,50.600,52.320,10.300,15.300,16.000,19.100,66,114,0.579,133,264,0.504,1,2,0.500,2.500,2.400,42.900,Sr,6-6,2.000,2.155,109.845,2.800,2023,45840,"Lexington, KY",,1.000,103.000,181.000,29.000,82.000,0.5691,0.3537,16.000,18.000,0.8889,,101.856,99.587,2.824,156.979,2.073,0.657,1.416,1.987,27.151,1.462,0.525,2.454,3.273,5.727,2.061,1.121,0.576,10.546,Wing F,0.140,1998-01-01,2023
2,DeJuan Clayton,California,P12,9,22.000,84.600,21.900,38.100,40.410,1.200,6.100,21.900,14.900,9,12,0.750,7,44,0.159,20,53,0.377,0.000,1.100,12.400,Sr,6-2,33.000,0.518,92.091,1.500,2023,46161,"Bowie, MD",,1.733,0.000,9.000,7.000,35.000,0.0000,0.2000,0.000,0.000,,,113.771,106.845,2.385,35.118,-4.191,-1.160,-3.031,-4.329,31.556,-1.452,-2.877,0.333,1.556,1.889,2.889,0.556,0.000,9.222,Pure PG,11.555,1997-05-18,2023


,column,dtype,nulls,null_pct
0,player,object,0,0.000
1,team,object,0,0.000
2,conf,object,0,0.000
3,gp,int64,0,0.000
4,min_per,float64,0,0.000
...,...,...,...,...
63,player_stat_64,float64,3,0.000
64,role,object,0,0.000
65,role_metric,float64,2,0.000
66,birth_date,object,0,0.000


In [88]:
if 'player_id' in player_multi_year.columns:
    id_summary = (
        player_multi_year
        .groupby('player_id', dropna=True)
        .agg(
            seasons=('source_year', 'nunique'),
            player_names=('player', lambda s: sorted(set(map(str, s.dropna())))[:5]),
            teams=('team', lambda s: sorted(set(map(str, s.dropna())))),
            first_year=('source_year', 'min'),
            last_year=('source_year', 'max'),
        )
        .reset_index()
    )
    display(id_summary.sort_values(['seasons', 'last_year'], ascending=False).head(20))
    display(id_summary.loc[id_summary['teams'].map(len) > 1].sort_values('seasons', ascending=False).head(20))
else:
    print('player_id column not available.')

,player_id,seasons,player_names,teams,first_year,last_year
119,65443,3,[Robby Carmody],"[Le Moyne, Mercer, Notre Dame]",2023,2025
168,65575,3,[K.J. Hymes],[Nevada],2023,2025
189,65627,3,[Jack Clark],"[Clemson, N.C. State, VCU]",2023,2025
227,65717,3,"[Robert Braswell, Robert Braswell IV]",[Charlotte],2023,2025
285,65852,3,[Wynston Tabbs],"[East Carolina, Morgan St.]",2023,2025
299,65878,3,[Tyrel Bladen],"[Norfolk St., Rider]",2023,2025
326,65938,3,[Keenan Fitzmorris],"[Northwestern, Stony Brook]",2023,2025
386,66085,3,[Kobe Julien],"[Arkansas St., Louisiana]",2023,2025
477,66286,3,[Messiah Jones],"[Towson, Wofford]",2023,2025
570,66493,3,[Savion Lewis],[Quinnipiac],2023,2025


,player_id,seasons,player_names,teams,first_year,last_year
3317,75059,3,[Rafael Castro],"[George Washington, Providence]",2023,2025
3133,74819,3,[Reyne Smith],"[Charleston, Louisville]",2023,2025
5043,77108,3,[Jace Whiting],"[Boise St., UNLV]",2023,2025
5051,77116,3,[Dorian Finister],"[Kansas St., Sam Houston St.]",2023,2025
3211,74919,3,[Denver Jones],"[Auburn, FIU]",2023,2025
5055,77120,3,[Travis Roberts],"[Jacksonville St., Marist]",2023,2025
3202,74906,3,[Kaleb Stewart],"[Louisiana Tech, South Dakota]",2023,2025
3191,74894,3,[Quentin Diboundje],"[East Carolina, Rhode Island]",2023,2025
3189,74891,3,[Will McClendon],"[San Jose St., UCLA]",2023,2025
3183,74884,3,[Zeke Mayo],"[Kansas, South Dakota St.]",2023,2025


---
## 6. Team Four Factors and Shooting Style

This is the likely starting point for team style/system features: eFG, turnover rate, offensive rebounding, free throw rate, 2P/3P rates, assist rate, and defensive versions.


In [89]:
four_factors = read_csv_endpoint(f'{SEASON}_fffinal.csv')
df_overview(four_factors, f'four_factors_{SEASON}')

GET https://barttorvik.com/2025_fffinal.csv (cache hit)
2025_fffinal.csv: header has 37 columns but data has 41; adding generated extra labels.
four_factors_2025: 364 rows x 41 columns


,TeamName,eFG%,Rk,eFG% Def,Rk_2,FTR,Rk_3,FTR Def,Rk_4,OR%,Rk_5,DR%,Rk_6,TO%,Rk_7,TO% Def.,Rk_8,3P%,rk,3pD%,rk_2,2p%,rk_3,2p%D,rk_4,ft%,rk_5,ft%D,rk_6,3P rate,rk_7,3P rate D,rk_8,arate,rk_9,arateD,rk_10,extra_01,extra_02,extra_03,extra_04
0,UNC Asheville,52.000,113,53.400,298,34.400,140,40.100,333,29.700,193,34.100,341,14.800,35,18.000,114,34.200,156,35.200,264,52.400,126,53.800,293,72.200,184,70.300,82,35.000,280,38.900,185,45.600,313,50.300,135,11.500,71,5.400,1
1,Bryant,50.700,175,47.800,49,29.900,274,30.000,103,33.800,59,29.500,162,16.300,126,16.400,226,33.000,216,32.500,98,51.300,161,47.300,45,71.600,210,72.000,176,34.700,288,36.100,89,48.700,262,43.200,19,14.700,10,5.700,2
2,Central Connecticut,52.200,106,47.700,43,25.900,345,18.900,2,26.500,274,25.300,14,17.200,180,18.100,110,32.900,219,30.600,24,53.700,81,49.000,98,74.500,99,68.300,14,34.800,282,42.000,288,50.400,223,52.600,205,12.100,50,6.000,3
3,Lafayette,50.200,206,50.400,155,30.500,255,26.900,38,24.000,330,28.000,89,16.600,141,16.700,199,34.000,168,35.800,301,49.700,240,48.300,77,69.200,280,73.500,262,40.300,145,37.900,143,59.400,37,54.300,244,9.900,141,6.500,4
4,Cornell,58.700,1,52.200,244,28.600,311,31.300,133,25.500,309,28.900,133,16.800,160,15.300,305,37.400,25,34.700,228,61.100,1,52.300,241,76.200,49,70.000,63,48.600,13,39.100,190,60.700,21,53.900,231,8.100,248,6.500,5


,column,dtype,nulls,null_pct
0,TeamName,object,0,0.000
1,eFG%,float64,0,0.000
2,Rk,int64,0,0.000
3,eFG% Def,float64,0,0.000
4,Rk_2,int64,0,0.000
5,FTR,float64,0,0.000
6,Rk_3,int64,0,0.000
7,FTR Def,float64,0,0.000
8,Rk_4,int64,0,0.000
9,OR%,float64,0,0.000


In [90]:
style_cols = [col for col in [
    'TeamName', 'eFG%', 'eFG% Def', 'FTR', 'FTR Def', 'OR%', 'DR%',
    'TO%', 'TO% Def.', '3P%', '3pD%', '2p%', '2p%D',
    '3P rate', '3P rate D', 'arate', 'arateD', 'extra_01', 'extra_03'
] if col in four_factors.columns]

four_factors[style_cols].head(20)


,TeamName,eFG%,eFG% Def,FTR,FTR Def,OR%,DR%,TO%,TO% Def.,3P%,3pD%,2p%,2p%D,3P rate,3P rate D,arate,arateD,extra_01,extra_03
0,UNC Asheville,52.000,53.400,34.400,40.100,29.700,34.100,14.800,18.000,34.200,35.200,52.400,53.800,35.000,38.900,45.600,50.300,11.500,5.400
1,Bryant,50.700,47.800,29.900,30.000,33.800,29.500,16.300,16.400,33.000,32.500,51.300,47.300,34.700,36.100,48.700,43.200,14.700,5.700
2,Central Connecticut,52.200,47.700,25.900,18.900,26.500,25.300,17.200,18.100,32.900,30.600,53.700,49.000,34.800,42.000,50.400,52.600,12.100,6.000
3,Lafayette,50.200,50.400,30.500,26.900,24.000,28.000,16.600,16.700,34.000,35.800,49.700,48.300,40.300,37.900,59.400,54.300,9.900,6.500
4,Cornell,58.700,52.200,28.600,31.300,25.500,28.900,16.800,15.300,37.400,34.700,61.100,52.300,48.600,39.100,60.700,53.900,8.100,6.500
5,Delaware St.,48.800,53.800,36.300,40.300,34.000,32.100,18.100,20.000,35.500,33.500,47.600,56.300,20.700,41.100,44.800,57.400,6.400,6.500
6,Loyola Marymount,50.200,48.500,29.000,28.900,22.800,30.900,15.200,15.700,33.100,31.800,50.600,49.000,40.400,36.600,50.900,51.000,10.400,6.600
7,UC Irvine,51.700,45.800,36.000,22.400,27.900,26.500,18.000,17.200,35.600,33.900,50.700,42.800,35.300,37.300,60.100,55.000,8.700,6.700
8,Maryland,53.600,47.100,33.100,26.100,30.700,28.200,14.300,20.100,37.300,30.600,52.200,47.800,35.400,36.200,49.100,49.200,11.800,6.800
9,Stanford,50.700,50.500,29.900,35.500,30.900,28.400,15.600,17.400,34.200,33.400,50.200,50.700,41.700,33.700,53.900,40.500,10.000,6.800


---
## 7. Game Results and Super Schedule

These datasets help connect team/player features to game outcomes, pre-game projections, tempo, and adjusted ratings around specific matchups.


In [91]:
game_results_raw = read_csv_endpoint(f'{SEASON}_results.csv', has_header=False, prefix='game_result')
game_results = apply_position_labels(game_results_raw, dict(enumerate(GAME_RESULTS_COLS, start=1)))
df_overview(game_results, f'game_results_{SEASON}')


GET https://barttorvik.com/2025_results.csv (cache hit)
game_results_2025: 6,307 rows x 11 columns


,game_id,date,prediction,team1_barthag,team2_barthag,team1_score,team2_score,team1_rank,team2_rank,game_barthag,spread
0,Davis & ElkinsMarshall11-4,11/4/24,Marshall (100%),56.680,90.855,57.000,90.000,0,216,73.226,-5.200
1,DrexelTemple11-12,11/12/24,"Temple -7.5, 75-68 (77%)",67.971,75.455,61.000,69.000,189,126,67.979,41.718
2,Youngstown St.Syracuse11-16,11/16/24,"Syracuse -14.6, 84-69 (88%)",69.041,83.640,95.000,104.000,202,84,74.247,41.779
3,Kent St.Auburn11-13,11/13/24,"Auburn -21.4, 85-64 (97%)",63.565,84.990,56.000,79.000,160,3,70.542,46.478
4,Loyola LASouthern Miss11-12,11/12/24,Southern Miss (100%),56.135,89.626,65.000,104.000,0,225,71.748,-6.425


,column,dtype,nulls,null_pct
0,game_id,object,0,0.000
1,date,object,0,0.000
2,prediction,object,0,0.000
3,team1_barthag,float64,0,0.000
4,team2_barthag,float64,0,0.000
5,team1_score,float64,0,0.000
6,team2_score,float64,0,0.000
7,team1_rank,int64,0,0.000
8,team2_rank,int64,0,0.000
9,game_barthag,float64,0,0.000


In [92]:
super_sked_raw = read_csv_endpoint(f'{SEASON}_super_sked.csv', has_header=False, prefix='super_sked')
super_sked = apply_position_labels(super_sked_raw, dict(enumerate(SUPER_SKED_COLS, start=1)))
df_overview(super_sked, f'super_sked_{SEASON}')


GET https://barttorvik.com/2025_super_sked.csv (cache hit)
super_sked_2025: 6,293 rows x 55 columns


,muid,date,conmatch,matchup,prediction,ttq,conf,venue,team1,t1oe,t1de,t1py,t1wp,t1propt,team2,t2oe,t2de,t2py,t2wp,t2propt,tpro,t1qual,t2qual,gp,result,tempo,possessions,t1pts,t2pts,winner,loser,t1adjt,t2adjt,t1adjo,t1adjd,t2adjo,t2adjd,gamevalue,mismatch,blowout,t1elite,t2elite,ord_date,t1ppp,t2ppp,gameppp,t1rk,t2rk,t1gs,t2gs,gamestats,overtimes,t1fun,t2fun,results
0,Northeastern St.Tulsa11-4,11/4/24,D2 at Amer,0 Northeastern St. at 243 Tulsa,Tulsa (100%),-8.437,99,0,Northeastern St.,76.986,125.612,0.004,0,50.803,Tulsa,103.426,106.776,0.409,1,81.460,66.519,,,1,"Tulsa, 82-68",72.550,72.550,68.000,82.000,Tulsa,Northeastern St.,73.625,72.550,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.906,1.000,739194,0.937,1.130,1.034,0,243,-,-,"['11/4/24', 200.0, 'Northeastern St.', 'Tulsa', 27.0, 58.0, 9.0, 24.0, 5.0, 10.0, 7.0, 28.0, 35....",,0.005,0.995,"['Northeastern St.Tulsa11-4', '11/4/24', 'Tulsa (100%)', '54.4666502', '94.01758258', '68', '82'..."
1,Gardner WebbTennessee11-4,11/4/24,BSth at SEC,271 Gardner Webb at 6 Tennessee,"Tennessee -31.6, 86-55 (99%)",39.708,0,0,Gardner Webb,101.063,112.458,0.226,0,54.544,Tennessee,121.720,90.683,0.967,1,86.177,65.689,Gardner Webb,,1,"Tennessee, 80-64",67.550,67.550,64.000,80.000,Tennessee,Gardner Webb,71.337,65.732,38.617,36.383,39.380,34.650,0.349,0.482,0.581,0.184,0.976,739194,0.947,1.184,1.066,271,6,0.6648947198524917,0.8132553050462586,"['11/4/24', 200.0, 'Gardner Webb', 'Tennessee', 24.0, 55.0, 6.0, 23.0, 10.0, 15.0, 10.0, 22.0, 3...",,0.010,0.990,"['Gardner WebbTennessee11-4', '11/4/24', 'Tennessee -23.4, 85-61 (98%)', '61.14491902', '84.5577..."
2,AvilaHouston Christian11-4,11/4/24,D2 at Slnd,0 Avila at 258 Houston Christian,Houston Christian (100%),-10.489,99,0,Avila,76.986,125.612,0.004,0,49.650,Houston Christian,101.992,106.766,0.371,1,78.132,64.817,,,1,"Houston Christian, 86-59",76.263,76.263,59.000,86.000,Houston Christian,Avila,79.424,76.263,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.919,1.000,739194,0.774,1.128,0.951,0,258,-,-,"['11/4/24', 200.0, 'Avila', 'Houston Christian', 24.0, 61.0, 4.0, 15.0, 7.0, 15.0, 6.0, 30.0, 36...",,0.006,0.994,"['AvilaHouston Christian11-4', '11/4/24', 'Houston Christian (100%)', '60.9837628', '86.61908423..."
3,Framingham St.Stonehill11-4,11/4/24,D2 at NEC,0 Framingham St. at 329 Stonehill,Stonehill (100%),-14.341,99,0,Framingham St.,76.986,125.612,0.004,0,53.625,Stonehill,101.810,113.549,0.222,1,78.451,65.442,,,1,"Stonehill, 81-62",72.875,72.875,62.000,81.000,Stonehill,Framingham St.,75.172,72.875,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.959,1.000,739194,0.851,1.111,0.981,0,329,-,-,"['11/4/24', 200.0, 'Framingham St.', 'Stonehill', 21.0, 50.0, 3.0, 7.0, 17.0, 19.0, 6.0, 24.0, 3...",,0.012,0.988,"['Framingham St.Stonehill11-4', '11/4/24', 'Stonehill (100%)', '58.00118346', '82.81721505', '62..."
4,ElonNorth Carolina11-4,11/4/24,CAA at ACC,235 Elon at 33 North Carolina,"North Carolina -24.4, 90-65 (96%)",39.598,0,0,Elon,105.107,113.597,0.290,0,65.158,North Carolina,120.861,98.526,0.913,1,89.600,68.546,Elon,,1,"North Carolina, 90-76",74.625,74.625,76.000,90.000,North Carolina,Elon,71.115,77.119,59.872,56.937,60.578,56.123,0.539,0.245,0.898,0.389,0.967,739194,1.018,1.206,1.112,235,33,0.6406161205884155,0.7064941804688802,"['11/4/24', 200.0, 'Elon', 'North Carolina', 26.0, 61.0, 13.0, 29.0, 11.0, 14.0, 10.0, 30.0, 40....",,0.038,0.962,"['ElonNorth Carolina11-4', '11/4/24', 'North Carolina -27.4, 91-63 (98%)', '63.35815342', '90.80..."


,column,dtype,nulls,null_pct
0,muid,object,0,0.000
1,date,object,0,0.000
2,conmatch,object,0,0.000
3,matchup,object,0,0.000
4,prediction,object,0,0.000
5,ttq,float64,0,0.000
6,conf,int64,0,0.000
7,venue,int64,0,0.000
8,team1,object,0,0.000
9,t1oe,float64,0,0.000


---
## 8. Team Game Log

Team-specific game log endpoint. Useful for validating one team's schedule and joining game-level features back to team ratings.


In [93]:
team_game_log = read_json_endpoint(
    'getgamestats.php',
    params={'year': SEASON, 'tvalue': SAMPLE_TEAM},
)
df_overview(team_game_log, f'{SAMPLE_TEAM}_game_log_{SEASON}')


GET https://barttorvik.com/getgamestats.php?year=2025&tvalue=Duke (cache hit)
Duke_game_log_2025: 39 rows x 31 columns


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30
0,3/29/25,3,Duke,ACC,Alabama,N,"W, 85-65",126.800,76.700,117.200,58.900,17.900,25.900,39.300,89.600,41.500,15.200,22.700,21.500,99.700,SEC,2,2025,72.500,AlabamaDuke3-29,Jon Scheyer,Nate Oats,8.548,0.961,"[""3/29/25"", 200, ""Alabama"", ""Duke"", 23, 65, 8, 32, 11, 14, 10, 20, 30, 14, 9, 5, 11, 18, 65, 30,...",7
1,3/27/25,3,Duke,ACC,Arizona,N,"W, 100-93",156.800,111.900,142.100,70.000,12.800,31.800,49.100,132.100,54.500,8.500,34.300,34.800,98.000,B12,2,2025,70.400,ArizonaDuke3-27,Jon Scheyer,Tommy Lloyd,5.986,0.946,"[""3/27/25"", 200, ""Arizona"", ""Duke"", 30, 66, 12, 26, 21, 23, 12, 15, 27, 10, 5, 1, 6, 24, 93, 33,...",5
2,11/8/24,0,Duke,ACC,Army,H,"W, 100-58",125.000,95.000,141.000,61.300,11.300,43.600,25.400,81.800,39.100,18.300,23.300,14.100,95.900,Pat,2,2025,70.900,ArmyDuke11-8,Jon Scheyer,Kevin Kuwik,15.682,0.152,"[""11/8/24"", 200, ""Army"", ""Duke"", 21, 64, 8, 29, 8, 9, 10, 22, 32, 11, 7, 2, 13, 15, 58, 35, 71, ...",5
3,12/4/24,0,Duke,ACC,Auburn,H,"W, 84-78",150.200,106.300,133.800,57.800,6.400,33.300,46.600,124.300,53.100,12.700,40.500,18.800,98.200,SEC,2,2025,62.800,AuburnDuke12-4,Jon Scheyer,Bruce Pearl,1.886,0.966,"[""12/4/24"", 200, ""Auburn"", ""Duke"", 29, 64, 10, 25, 10, 12, 15, 24, 39, 17, 4, 3, 8, 20, 78, 29, ...",4
4,3/23/25,3,Duke,ACC,Baylor,N,"W, 89-66",155.300,101.300,150.000,77.800,10.100,17.600,51.100,111.200,42.600,8.400,40.900,14.700,99.300,B12,2,2025,59.300,BaylorDuke3-23,Jon Scheyer,Scott Drew,8.806,0.899,"[""3/23/25"", 200, ""Baylor"", ""Duke"", 25, 68, 8, 25, 8, 10, 18, 14, 32, 9, 2, 0, 5, 16, 66, 29, 45,...",4


,column,dtype,nulls,null_pct
0,0,object,0,0.000
1,1,int64,0,0.000
2,2,object,0,0.000
3,3,object,0,0.000
4,4,object,0,0.000
5,5,object,0,0.000
6,6,object,0,0.000
7,7,float64,0,0.000
8,8,float64,0,0.000
9,9,float64,0,0.000


---
## 9. Team Shooting Splits from PBP-Derived File

This endpoint is promising for scheme fit because it can expose close 2s, long 2s, threes, free throws, possessions, and defensive equivalents without scraping play-by-play directly.


In [94]:
pbp_text = fetch_text(f'{SEASON}_pbp_teamsstats.json')
pbp_raw = json.loads(pbp_text)
print(type(pbp_raw))
print(pbp_raw.keys() if isinstance(pbp_raw, dict) else 'not a dict')

GET https://barttorvik.com/2025_pbp_teamsstats.json (cache hit)
<class 'dict'>
dict_keys(['offense', 'dunks', 'defense', 'ddunks'])


In [95]:
# Flatten nested dicts into a wide team table. Section labels are from Bart/pybart docs.
if isinstance(pbp_raw, dict):
    flattened_parts = []
    for section_name, section_values in pbp_raw.items():
        part = pd.DataFrame.from_dict(section_values, orient='index')
        labels = PBP_SECTION_COLS.get(section_name)
        if labels is not None and len(labels) == part.shape[1]:
            part.columns = [f'{section_name}_{label}' for label in labels]
        else:
            part = part.add_prefix(f'{section_name}_')
        flattened_parts.append(part)
    team_shooting = pd.concat(flattened_parts, axis=1).reset_index().rename(columns={'index': 'team'})
    df_overview(team_shooting, f'team_shooting_splits_{SEASON}')
else:
    print('Unexpected PBP team stats shape.')


team_shooting_splits_2025: 364 rows x 25 columns


,team,offense_close_2_att,offense_close_2_made,offense_long_2_att,offense_long_2_made,offense_three_att,offense_three_made,offense_ft_att,offense_ft_made,offense_possessions,dunks_dunk_att,dunks_dunk_made,dunks_dunk_share,defense_opp_close_2_att,defense_opp_close_2_made,defense_opp_long_2_att,defense_opp_long_2_made,defense_opp_three_att,defense_opp_three_made,defense_opp_ft_att,defense_opp_ft_made,defense_opp_possessions,ddunks_opp_dunk_att,ddunks_opp_dunk_made,ddunks_opp_dunk_share
0,UC Santa Barbara,342,206,171,146,238,38,320,520,267,29,3,22,377,257,142,200,308,49,206,463,174,24,0,16
1,Arkansas Pine Bluff,457,343,198,94,154,32,198,441,170,24,7,14,461,229,224,108,175,45,283,509,244,85,6,63
2,Portland St.,499,332,207,99,227,29,172,366,141,93,18,64,402,289,162,112,194,32,203,437,186,46,8,36
3,Indiana,461,262,270,207,340,71,203,429,175,81,3,73,376,268,173,209,310,64,242,499,196,56,10,38
4,Louisiana,351,294,113,160,331,37,219,480,182,26,6,17,445,283,185,146,225,43,202,440,165,63,10,49


,column,dtype,nulls,null_pct
0,team,object,0,0.000
1,offense_close_2_att,int64,0,0.000
2,offense_close_2_made,int64,0,0.000
3,offense_long_2_att,int64,0,0.000
4,offense_long_2_made,int64,0,0.000
5,offense_three_att,int64,0,0.000
6,offense_three_made,int64,0,0.000
7,offense_ft_att,int64,0,0.000
8,offense_ft_made,int64,0,0.000
9,offense_possessions,int64,0,0.000


---
## 10. Regular Season Team Slice

This is useful for pre-tournament or regular-season-only modeling. `type=R` means regular season in Bart's comments.


In [96]:
regular_season_slice = read_json_endpoint(
    'teamslicejson.php',
    params={'year': SEASON, 'json': 1, 'type': 'R'},
)
df_overview(regular_season_slice, f'regular_season_slice_{SEASON}')

GET https://barttorvik.com/teamslicejson.php?year=2025&json=1&type=R (cache hit)
regular_season_slice_2025: 364 rows x 37 columns


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36
0,UC Santa Barbara,108.045,107.563,0.513,19–13,19,32,54.600,48.900,29.700,32.000,18.200,17.200,27.500,29.200,66.500,52.400,50.500,38.100,30.800,8.400,7.100,58.300,46.100,47.400,36.900,66.000,,,,2025,,,,-8.072,77.200,77.400
1,Arkansas Pine Bluff,95.999,122.272,0.058,4–25,4,29,50.300,56.300,32.800,40.500,20.400,16.800,26.300,34.600,72.500,52.600,58.500,31.000,35.700,6.300,9.200,52.200,59.500,37.900,44.900,72.300,,,,2025,,,,-20.010,59.000,69.600
2,Toledo,110.828,117.345,0.341,17–15,17,32,50.500,56.400,35.500,26.500,13.700,15.500,29.800,30.900,69.500,50.600,55.900,33.600,38.200,7.100,7.500,46.000,45.800,28.600,37.500,69.000,,,,2025,,,,-8.824,75.700,74.200
3,Indiana,112.796,97.730,0.839,19–13,19,32,51.100,49.800,33.100,29.600,16.700,16.400,31.500,28.100,69.100,52.600,50.300,32.100,32.700,9.200,8.200,58.400,51.800,33.200,38.900,68.700,,,,2025,,,,0.950,70.600,72.500
4,Louisiana,98.373,110.094,0.215,11–21,11,32,45.700,51.300,29.800,43.800,16.000,19.100,25.900,35.500,66.400,45.000,53.800,31.300,31.500,8.500,11.500,45.300,49.200,38.100,36.900,67.000,,,,2025,,,,-15.630,72.900,73.000


,column,dtype,nulls,null_pct
0,0,object,0,0.000
1,1,float64,0,0.000
2,2,float64,0,0.000
3,3,float64,0,0.000
4,4,object,0,0.000
5,5,int64,0,0.000
6,6,int64,0,0.000
7,7,float64,0,0.000
8,8,float64,0,0.000
9,9,float64,0,0.000


---
## 11. Player Game Stats

The player-game endpoint is the next best source for talent uncertainty, game-to-game variance, role changes, and early Bayesian adjusted plus-minus experiments.

Observed from 2023-2025:

- `2023_all_advgames.json.gz`: 112,744 rows x 53 columns
- `2024_all_advgames.json.gz`: 113,103 rows x 53 columns
- `2025_all_advgames.json.gz`: 114,218 rows x 53 columns
- 2025 response was about 44 MB over the wire before caching

Only the positions below are labeled for now. Leave the rest positional until we verify them from source documentation or value checks.


In [97]:
PLAYER_GAME_POSITION_LABELS = {
    1: 'game_date_id',
    2: 'game_date',
    6: 'opponent',
    7: 'game_id',
    9: 'minutes',
    10: 'game_ortg',
    11: 'usage',
    12: 'efg',
    13: 'ts_pct',
    14: 'or_pct',
    15: 'dr_pct',
    16: 'ast_pct',
    17: 'to_pct',
    18: 'dunk_made',
    19: 'dunk_att',
    20: 'close_2_made',
    21: 'close_2_att',
    22: 'far_2_made',
    23: 'far_2_att',
    24: 'two_made',
    25: 'two_att',
    26: 'three_made',
    27: 'three_att',
    28: 'ft_made',
    29: 'ft_att',
    34: 'points',
    35: 'off_reb',
    36: 'def_reb',
    37: 'assists',
    38: 'turnovers',
    39: 'steals',
    40: 'blocks',
    47: 'venue',
    48: 'team',
    49: 'player',
    50: 'height_inches',
    51: 'yr',
    52: 'player_id',
    53: 'season',
}

PLAYER_GAME_YEARS = [2023, 2024, 2025]
LOAD_PLAYER_GAME_STATS = True

if LOAD_PLAYER_GAME_STATS:
    player_game_years = []
    for year in PLAYER_GAME_YEARS:
        raw = json.loads(fetch_text(f'{year}_all_advgames.json.gz', timeout=90))
        df_raw = pd.DataFrame(raw, columns=generated_columns(len(raw[0]), prefix='player_game'))
        df = apply_position_labels(coerce_numeric_columns(df_raw), PLAYER_GAME_POSITION_LABELS)
        df['source_year'] = year
        player_game_years.append(df)
    player_game_stats = pd.concat(player_game_years, ignore_index=True)
    df_overview(player_game_stats, 'player_game_stats_2023_2025', n=3)
    display(player_game_stats[[col for col in [
        'source_year', 'game_date', 'game_id', 'team', 'opponent', 'venue',
        'player', 'player_id', 'yr', 'height_inches', 'minutes', 'game_ortg',
        'usage', 'efg', 'ts_pct', 'or_pct', 'dr_pct', 'ast_pct', 'to_pct',
        'points', 'two_made', 'two_att', 'three_made', 'three_att', 'ft_made', 'ft_att'
    ] if col in player_game_stats.columns]].head(20))
else:
    print('Set LOAD_PLAYER_GAME_STATS = True to load cached/fetched player-game files for 2023-2025.')


GET https://barttorvik.com/2023_all_advgames.json.gz (cache hit)
GET https://barttorvik.com/2024_all_advgames.json.gz (cache hit)
GET https://barttorvik.com/2025_all_advgames.json.gz (cache hit)
player_game_stats_2023_2025: 340,065 rows x 54 columns


,game_date_id,game_date,player_game_03,player_game_04,player_game_05,opponent,game_id,player_game_08,minutes,game_ortg,usage,efg,ts_pct,or_pct,dr_pct,ast_pct,to_pct,player_game_18,player_game_19,player_game_20,player_game_21,player_game_22,player_game_23,player_game_24,player_game_25,player_game_26,player_game_27,player_game_28,player_game_29,player_game_30,player_game_31,player_game_32,player_game_33,player_game_34,player_game_35,player_game_36,player_game_37,player_game_38,player_game_39,player_game_40,player_game_41,player_game_42,player_game_43,player_game_44,player_game_45,player_game_46,venue,team,player,height_inches,yr,player_id,season,source_year
0,20221107,11-7,0,0,1,North Dakota St.,North Dakota St.Arkansas11-7,1,39.000,78.700,28.200,42.900,50.600,0.000,15.000,0.000,24.700,0,0,3,4,3,8,6.000,12.000,0.000,2.000,6.000,8.000,-2.400,-3.587,1.174,-2.900,18,0.000,6.000,0.000,5.000,2.000,1.000,2.800,2.900,1.000,73.425,-2.413,-2.916,H,Arkansas,Davonte Davis,76.000,Jr,73528,2023,2023
1,20221111,11-11,0,0,1,Fordham,FordhamArkansas11-11,1,29.000,82.400,20.500,25.000,27.000,0.000,13.300,26.700,9.600,0,0,1,3,2,6,3.000,9.000,0.000,3.000,1.000,2.000,3.200,-2.503,5.704,0.400,7,0.000,3.000,5.000,1.000,3.000,0.000,5.700,0.000,1.000,72.312,3.201,0.394,H,Arkansas,Davonte Davis,76.000,Jr,73528,2023,2023
2,20221116,11-16,0,0,1,South Dakota St.,South Dakota St.Arkansas11-16,1,33.000,116.600,15.500,60.000,59.400,0.000,6.200,16.600,9.900,0,0,1,2,2,3,3.000,5.000,2.000,5.000,1.000,2.000,6.100,3.675,2.459,2.800,13,0.000,2.000,3.000,1.000,1.000,0.000,1.500,0.000,1.000,79.312,6.134,2.779,H,Arkansas,Davonte Davis,76.000,Jr,73528,2023,2023


,column,dtype,nulls,null_pct
0,game_date_id,int64,0,0.000
1,game_date,object,0,0.000
2,player_game_03,int64,0,0.000
3,player_game_04,int64,0,0.000
4,player_game_05,int64,0,0.000
5,opponent,object,0,0.000
6,game_id,object,0,0.000
7,player_game_08,int64,0,0.000
8,minutes,float64,0,0.000
9,game_ortg,float64,0,0.000


,source_year,game_date,game_id,team,opponent,venue,player,player_id,yr,height_inches,minutes,game_ortg,usage,efg,ts_pct,or_pct,dr_pct,ast_pct,to_pct
0,2023,11-7,North Dakota St.Arkansas11-7,Arkansas,North Dakota St.,H,Davonte Davis,73528,Jr,76.000,39.000,78.700,28.200,42.900,50.600,0.000,15.000,0.000,24.700
1,2023,11-11,FordhamArkansas11-11,Arkansas,Fordham,H,Davonte Davis,73528,Jr,76.000,29.000,82.400,20.500,25.000,27.000,0.000,13.300,26.700,9.600
2,2023,11-16,South Dakota St.Arkansas11-16,Arkansas,South Dakota St.,H,Davonte Davis,73528,Jr,76.000,33.000,116.600,15.500,60.000,59.400,0.000,6.200,16.600,9.900
3,2023,11-21,ArkansasLouisville11-21,Arkansas,Louisville,N,Davonte Davis,73528,Jr,76.000,32.000,96.000,10.100,25.000,25.000,0.000,0.000,22.500,17.700
4,2023,11-22,ArkansasCreighton11-22,Arkansas,Creighton,N,Davonte Davis,73528,Jr,76.000,38.000,106.500,11.300,38.900,45.200,0.000,12.100,7.100,0.000
5,2023,11-23,ArkansasSan Diego St.11-23,Arkansas,San Diego St.,N,Davonte Davis,73528,Jr,76.000,17.000,61.500,14.300,25.000,25.000,7.200,7.200,12.400,22.100
6,2023,12-3,San Jose St.Arkansas12-3,Arkansas,San Jose St.,H,Davonte Davis,73528,Jr,76.000,22.000,92.600,6.000,0.000,0.000,0.000,4.900,14.700,0.000
7,2023,12-6,UNC GreensboroArkansas12-6,Arkansas,UNC Greensboro,H,Davonte Davis,73528,Jr,76.000,29.000,61.400,28.900,23.100,33.600,3.500,26.400,0.000,19.400
8,2023,12-10,ArkansasOklahoma12-10,Arkansas,Oklahoma,N,Davonte Davis,73528,Jr,76.000,24.000,107.100,14.800,25.000,39.200,6.000,6.900,10.000,0.000
9,2023,12-17,BradleyArkansas12-17,Arkansas,Bradley,H,Davonte Davis,73528,Jr,76.000,30.000,98.700,19.000,38.900,38.900,12.900,19.800,19.000,10.000


---
## 11b. Player-Game Column Audit

The full player-game endpoint is now loaded for 2023-2025. The labels below are split into two groups: fields validated by basketball identities or season-total reconciliation, and fields left as positional `player_game_*` columns until we can verify their meaning from source examples.


In [ ]:
if 'player_game_stats' in globals():
    positional_cols = [col for col in player_game_stats.columns if col.startswith('player_game_')]
    audit_rows = []
    for col in positional_cols:
        s = player_game_stats[col]
        numeric = pd.to_numeric(s, errors='coerce')
        numeric_non_null = numeric.notna().sum()
        sample_values = s.drop_duplicates().head(8).tolist()
        audit_rows.append({
            'column': col,
            'dtype': str(s.dtype),
            'non_null': int(s.notna().sum()),
            'null_pct': float(s.isna().mean()),
            'n_unique': int(s.nunique(dropna=False)),
            'min': numeric.min() if numeric_non_null else np.nan,
            'max': numeric.max() if numeric_non_null else np.nan,
            'integerish': bool(((numeric.dropna() % 1) == 0).all()) if numeric_non_null else False,
            'sample_values': sample_values,
        })
    player_game_column_audit = pd.DataFrame(audit_rows)
    print(f'Unlabeled positional player-game columns: {len(player_game_column_audit)}')
    display(player_game_column_audit)
else:
    print('player_game_stats is not loaded. Set LOAD_PLAYER_GAME_STATS = True and rerun the player-game cell.')


In [ ]:
if 'player_game_stats' in globals():
    pg = player_game_stats.copy()
    checks = [
        {
            'check': 'dunks are a subset of close 2s',
            'pass_rate': ((pg['dunk_made'] <= pg['dunk_att']) & (pg['dunk_att'] <= pg['close_2_att']) & (pg['dunk_made'] <= pg['close_2_made'])).mean(),
        },
        {
            'check': 'two_made = close_2_made + far_2_made',
            'pass_rate': (pg['two_made'] == pg['close_2_made'] + pg['far_2_made']).mean(),
        },
        {
            'check': 'two_att = close_2_att + far_2_att',
            'pass_rate': (pg['two_att'] == pg['close_2_att'] + pg['far_2_att']).mean(),
        },
        {
            'check': 'points = 2*two_made + 3*three_made + ft_made',
            'pass_rate': (pg['points'] == 2 * pg['two_made'] + 3 * pg['three_made'] + pg['ft_made']).mean(),
        },
    ]
    box_score_identity_checks = pd.DataFrame(checks)
    display(box_score_identity_checks)

    season_box_totals = (
        pg[pg['source_year'] == SEASON]
        .groupby(['player_id', 'team', 'season'], as_index=False)
        .agg(
            two_made_pg=('two_made', 'sum'),
            two_att_pg=('two_att', 'sum'),
            three_made_pg=('three_made', 'sum'),
            three_att_pg=('three_att', 'sum'),
            ft_made_pg=('ft_made', 'sum'),
            ft_att_pg=('ft_att', 'sum'),
            points_pg=('points', 'sum'),
            games_pg=('game_id', 'nunique'),
        )
    )
    season_box_validation = season_box_totals.merge(
        player_stats[[
            'player_id', 'team', 'year', 'player', 'fg2m', 'fg2a', 'fg3m', 'fg3a', 'ftm', 'fta', 'gp'
        ]],
        left_on=['player_id', 'team', 'season'],
        right_on=['player_id', 'team', 'year'],
        how='left',
    )
    validation_pairs = {
        'two_made': ('two_made_pg', 'fg2m'),
        'two_att': ('two_att_pg', 'fg2a'),
        'three_made': ('three_made_pg', 'fg3m'),
        'three_att': ('three_att_pg', 'fg3a'),
        'ft_made': ('ft_made_pg', 'ftm'),
        'ft_att': ('ft_att_pg', 'fta'),
    }
    validation_summary = []
    matched = season_box_validation.dropna(subset=['year'])
    for label, (pg_col, season_col) in validation_pairs.items():
        validation_summary.append({
            'field': label,
            'matched_players': len(matched),
            'exact_match_rate': (matched[pg_col] == matched[season_col]).mean(),
            'total_abs_delta': (matched[pg_col] - matched[season_col]).abs().sum(),
        })
    display(pd.DataFrame(validation_summary))
    display(season_box_validation.sort_values('points_pg', ascending=False).head(10)[[
        'player', 'team', 'player_id', 'games_pg', 'gp',
        'two_made_pg', 'fg2m', 'two_att_pg', 'fg2a',
        'three_made_pg', 'fg3m', 'three_att_pg', 'fg3a',
        'ft_made_pg', 'ftm', 'ft_att_pg', 'fta', 'points_pg'
    ]])
else:
    print('player_game_stats is not loaded. Set LOAD_PLAYER_GAME_STATS = True and rerun the player-game cell.')


---
## 12. Cross-Source Team IDs

BartTorvik files appear to be team-name keyed rather than team-ID keyed. For cross-source joins, use team names only to build a one-time crosswalk, then use a canonical ID.

Practical EDA recommendation:

1. Use ESPN/CBBpy team IDs where they are available.
2. Map BartTorvik team names to those IDs through a reviewed crosswalk.
3. Keep unresolved/new schools as explicit unmatched rows with internal IDs until verified.

A normalized-name join plus a small alias map matched 361 of 364 BartTorvik 2025 teams to ESPN's public NCAAM team metadata in a local smoke test. The remaining unresolved teams were `Lindenwood`, `Queens`, and `Southern Indiana`, which appear to be newer/edge teams missing from the ESPN endpoint response.



In [98]:
import re
ESPN_TEAMS_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams?limit=400'
MANUAL_BART_TO_ESPN_ALIASES = {
    'Mississippi': 'Ole Miss',
    'Connecticut': 'UConn',
    'N.C. State': 'NC State',
    'McNeese St.': 'McNeese',
    'St. Thomas': 'St. Thomas-Minnesota',
    'Nebraska Omaha': 'Omaha',
    'Texas A&M Corpus Chris': 'Texas A&M-Corpus Christi',
    'Seattle': 'Seattle U',
    'Nicholls St.': 'Nicholls',
    'Sam Houston St.': 'Sam Houston',
    'San Jose St.': 'San José State',
    'Cal Baptist': 'California Baptist',
    'Appalachian St.': 'App State',
    'Miami FL': 'Miami',
    'Illinois Chicago': 'UIC',
    'Southeastern Louisiana': 'SE Louisiana',
    'UMKC': 'Kansas City',
    'Hawaii': "Hawai'i",
    'Albany': 'UAlbany',
    'LIU': 'Long Island University',
    'Tennessee Martin': 'UT Martin',
    'Grambling St.': 'Grambling',
    'USC Upstate': 'South Carolina Upstate',
    'Louisiana Monroe': 'UL Monroe',
}
def normalize_team_name(value: str) -> str:
    value = str(value).lower().replace('&', 'and')
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    words = ['state' if word == 'st' else word for word in value.split()]
    return ' '.join(words)
def fetch_external_json(url: str) -> dict:
    cache_path = cache_path_for_url(url)
    if CACHE_ENABLED and cache_path.exists():
        print('GET', url, '(cache hit)')
        text = cache_path.read_text()
    else:
        response = requests.get(url, headers=HEADERS, timeout=30)
        print('GET', response.url)
        print('status:', response.status_code, '| content-type:', response.headers.get('content-type'))
        response.raise_for_status()
        text = response.text
        if CACHE_ENABLED:
            CACHE_DIR.mkdir(parents=True, exist_ok=True)
            cache_path.write_text(text)
    return json.loads(text)
espn_raw = fetch_external_json(ESPN_TEAMS_URL)
espn_rows = []
for sport in espn_raw.get('sports', []):
    for league in sport.get('leagues', []):
        for item in league.get('teams', []):
            team = item.get('team', item)
            espn_rows.append({
                'espn_team_id': team.get('id'),
                'espn_uid': team.get('uid'),
                'espn_slug': team.get('slug'),
                'espn_abbreviation': team.get('abbreviation'),
                'espn_location': team.get('location'),
                'espn_short_name': team.get('shortDisplayName'),
                'espn_display_name': team.get('displayName'),
            })
espn_teams = pd.DataFrame(espn_rows)
espn_candidates = []
for _, row in espn_teams.iterrows():
    for field in ['espn_location', 'espn_short_name', 'espn_display_name']:
        espn_candidates.append({
            'match_key': normalize_team_name(row[field]),
            'match_field': field,
            **row.to_dict(),
        })
espn_candidates = pd.DataFrame(espn_candidates).drop_duplicates(['match_key', 'espn_team_id'])
bart_teams = team_ratings[['team', 'conf']].drop_duplicates().rename(columns={'team': 'bart_team', 'conf': 'bart_conf'})
bart_teams['match_name'] = bart_teams['bart_team'].replace(MANUAL_BART_TO_ESPN_ALIASES)
bart_teams['match_key'] = bart_teams['match_name'].map(normalize_team_name)
team_crosswalk = bart_teams.merge(espn_candidates, on='match_key', how='left')
team_crosswalk['canonical_team_id'] = np.where(
    team_crosswalk['espn_team_id'].notna(),
    'espn_' + team_crosswalk['espn_team_id'].astype(str),
    'bart_unmatched_' + team_crosswalk['match_key'].str.replace(' ', '_', regex=False),
)
matched_count = team_crosswalk.loc[team_crosswalk['espn_team_id'].notna(), 'bart_team'].nunique()
unmatched = team_crosswalk.loc[team_crosswalk['espn_team_id'].isna(), ['bart_team', 'bart_conf', 'match_name']].drop_duplicates()
print(f'Bart teams: {bart_teams["bart_team"].nunique()}')
print(f'ESPN teams: {espn_teams["espn_team_id"].nunique()}')
print(f'Matched Bart teams to ESPN IDs: {matched_count}')
print(f'Unmatched Bart teams: {len(unmatched)}')
display(team_crosswalk[['canonical_team_id', 'bart_team', 'bart_conf', 'espn_team_id', 'espn_location', 'espn_short_name', 'match_field']].head(20))
display(unmatched.sort_values('bart_team'))


GET https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams?limit=400 (cache hit)
Bart teams: 364
ESPN teams: 362
Matched Bart teams to ESPN IDs: 361
Unmatched Bart teams: 3


,canonical_team_id,bart_team,bart_conf,espn_team_id,espn_location,espn_short_name,match_field
0,espn_248,Houston,B12,248,Houston,Houston,espn_location
1,espn_150,Duke,ACC,150,Duke,Duke,espn_location
2,espn_2,Auburn,SEC,2,Auburn,Auburn,espn_location
3,espn_57,Florida,SEC,57,Florida,Florida,espn_location
4,espn_333,Alabama,SEC,333,Alabama,Alabama,espn_location
5,espn_2633,Tennessee,SEC,2633,Tennessee,Tennessee,espn_location
6,espn_2641,Texas Tech,B12,2641,Texas Tech,Texas Tech,espn_location
7,espn_2250,Gonzaga,WCC,2250,Gonzaga,Gonzaga,espn_location
8,espn_120,Maryland,B10,120,Maryland,Maryland,espn_location
9,espn_12,Arizona,B12,12,Arizona,Arizona,espn_location


,bart_team,bart_conf,match_name
321,Lindenwood,OVC,Lindenwood
207,Queens,ASun,Queens
335,Southern Indiana,OVC,Southern Indiana


---
## 13. Manual Website Scrape Fallback

Use only when a specific table is not available through direct data files. If the response shows browser verification, stop and look for a bulk endpoint instead.


In [99]:
def fetch_html_tables(path: str, params: dict | None = None) -> list[pd.DataFrame]:
    text = fetch_text(path, params=params)
    soup = BeautifulSoup(text, 'html.parser')
    print('html title:', soup.title.string if soup.title else None)
    return pd.read_html(StringIO(text))

RUN_HTML_FALLBACK = False

if RUN_HTML_FALLBACK:
    tables = fetch_html_tables('trank.php', params={'year': SEASON})
    print('tables found:', len(tables))
    if tables:
        df_overview(tables[0], 'html_table_0')
else:
    print('HTML fallback disabled. Prefer flat-file endpoints.')

HTML fallback disabled. Prefer flat-file endpoints.


---
## 14. PortalPoint Feature Mapping Notes

| PortalPoint need | BartTorvik candidate source | EDA question |
| --- | --- | --- |
| Team quality baseline | `{year}_team_results.csv` | Are ratings stable across seasons and are names easy to join? |
| Team style / scheme | `{year}_fffinal.csv`, `{year}_pbp_teamsstats.json`, `adjt` | Which variables best represent pace, shot profile, and ball movement? |
| Player production | `getadvstats.php?year={year}&csv=1`, `{year}_all_advgames.json.gz` | Which stats are complete enough for rotation players and game-level talent signals? |
| Player style | `getadvstats` shot-location columns, player-game shot mix, usage | Are player shot-location fields complete and interpretable? |
| Player role | `min_per`, `min_per_game`, `usage`, `role`, `yr`, `ht`, player-game stats | Can we separate role archetypes without play-by-play? |
| Transfer modeling | multi-year `player_id`, team changes, player names | Do IDs persist across schools/seasons? |
| Outcome validation | `{year}_results.csv`, `{year}_super_sked.csv` | What game/team outcome labels are easiest to join? |

EDA status after the player-game load:

| Question | Status | Notes |
| --- | --- | --- |
| Verify player-stat positional labels across 2023-2025 | Mostly answered | `getadvstats.php` returned 67 columns for each year. We verified key positions for identity, usage, shooting, BPM, role, and birth date. Unknown middle/tail fields remain as positional labels. |
| Confirm player ID stability across 2023-2025 | Answered | `player_id` is complete and stable enough to track multi-year players and transfers. Some names vary, so use `player_id` as the join key. |
| Quantify missingness for core player fields | Answered for season-level stats | 2025 has 5,060 unique players. Core identity/team/conference/year/role/minutes/usage/ORTG/BPM fields are effectively complete. `min_per_game` has 2 missing values; `role_metric` has 1 missing value in 2025. |
| Load player-game stats for 2023-2025 | Answered | The heavy endpoint loaded from cache as 340,065 player-game rows. The only meaningful missingness found so far is a tiny number of missing `height_inches` values. |
| Validate player-game box-score labels | Partially answered | Shot-location/2PT/3PT/FT/points columns now have formula checks and season-total reconciliation. Remaining impact-style columns stay positional until verified. |
| Identify useful player archetype features | Partially answered | Bart gives role labels, usage, efficiency, shot-location counts, height/class, and BPM. This is enough for first-pass archetypes, but role labels should be treated as inputs/priors rather than ground truth. |
| Identify talent-rating inputs | Partially answered | Treat this as our own Bayesian adjusted box plus-minus: Bart BPM/ORTG/usage/minutes/shot profile are priors/features, player-game stats provide variance and opponent/role context, and CBBpy/PBP may add possession-level signal. |
| Team style / scheme inputs | Partially answered | `team_results`, `fffinal`, and `pbp_teamsstats` give tempo, four factors, shot mix, and assist-rate style signals. Team game logs and regular-season slices still need friendlier labels. |
| Outcome validation data | Answered for game/team level | `results.csv` and `super_sked.csv` load cleanly and provide game outcomes/projections. |
| Team-ID crosswalk across sources | Partially answered | Bart appears name-keyed; ESPN provides team IDs for most programs. Use names only to build/review `team_crosswalk`, then join on `canonical_team_id`. |
| Need CBBpy PBP? | Open but likely yes | Bart is strong for model-ready priors and player-game aggregates. CBBpy/PBP is still likely needed for possession-level role context, on/off or lineup estimates, and custom Bayesian playing-time/talent models. |
| Cache heavy endpoints | Addressed in notebook | `fetch_text` caches responses under `.torvik_cache/`, which is gitignored. The player-game endpoint is large enough that caching is important. |

Next EDA tasks:
1. Use the new player-game audit cells to label only fields that pass validation or can be tied to source examples.
2. Review unresolved `team_crosswalk` rows and add/confirm aliases before joining to CBBpy/ESPN data.
3. Decide whether the first talent estimate starts as a Bart BPM/gBPM prior or as a new latent score estimated from player-game observations.
4. Test whether CBBpy play-by-play has enough substitution/lineup information for on/off and role-context estimates.
5. Add a small feature engineering section for archetype inputs, Bayesian talent inputs, and team-style/context inputs.
